# Day 24: Video Generation with Stable Video Diffusion

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
import requests
from io import BytesIO
import numpy as np

## 1. Load Stable Video Diffusion pipeline

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to(device)
pipe.enable_model_cpu_offload()  # saves VRAM
print("Pipeline loaded")

## 2. Load an input image and generate video

In [ ]:
# Use a landscape image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beach.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
image = image.resize((512, 512))
display(image)

# Generate video frames
with torch.no_grad():
    frames = pipe(image, decode_chunk_size=8, num_frames=25).frames[0]
print(f"Generated {len(frames)} frames")

## 3. Save and view video

In [ ]:
export_to_video(frames, "svd_output.mp4", fps=7)
print("Video saved as svd_output.mp4")
# To display in notebook (if using Colab or Jupyter with video support):
from IPython.display import Video
Video("svd_output.mp4", width=512)

## 4. Create a looping animation
Stable Video Diffusion generates a forward motion. To make it loop, we can reverse the frames and append, or blend start and end.

In [ ]:
# Simple loop: forward + reverse
looping_frames = frames + frames[::-1]
export_to_video(looping_frames, "svd_loop.mp4", fps=7)
Video("svd_loop.mp4", width=512)

## 5. Adjust motion magnitude (optional)
SVD has a `motion_bucket_id` parameter (default 127). Lower = less motion, higher = more motion.

In [ ]:
# Example: try a different motion bucket
with torch.no_grad():
    frames_slow = pipe(image, decode_chunk_size=8, num_frames=25, motion_bucket_id=40).frames[0]
    frames_fast = pipe(image, decode_chunk_size=8, num_frames=25, motion_bucket_id=200).frames[0]

export_to_video(frames_slow, "svd_slow.mp4", fps=7)
export_to_video(frames_fast, "svd_fast.mp4", fps=7)
print("Compare motion levels")